In [1]:
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
import os 
gemini_api_key = os.getenv("GEMINI_KEY")

model_client = OpenAIChatCompletionClient(model = "gemini-2.0-flash",api_key = gemini_api_key)


In [2]:
assistant = AssistantAgent(
    name = "text_agent",
    model_client = model_client,
    description = "you are a great text agent who can answer questions and help with tasks",
    system_message = "you are a great text agent who can answer questions in one or two sentences"
)


In [3]:
from autogen_agentchat.messages import TextMessage
async def text_agent():
    text_msg = TextMessage(content = "what is the capital of North Korea?",source = "user")
    result = await assistant.run(task = text_msg)
    print(result.messages[-1].content)


In [4]:
await text_agent()

The capital of North Korea is Pyongyang.



In [5]:
# Multimodal Agent
from autogen_agentchat.messages import TextMessage,MultiModalMessage
from autogen_core import Image as AGImage
import requests
from PIL import Image 
from io import BytesIO



In [16]:

async def multi_modal_agent(url,text):
    response = requests.get(url)
    image = Image.open(BytesIO(response.content))
    image = AGImage(image = image)

    multimodal_msg = MultiModalMessage(
        source = "user",
        content = [text,image]
    )
    result  = await assistant.run(task = multimodal_msg)
    print(result.messages[-1].content)



In [17]:
url = "https://picsum.photos/id/100/200/300"
text = "what is this image about?"
await multi_modal_agent(url,text)




The image shows a crowded beach with people in the water and on the sand. The weather appears hazy or foggy.



In [18]:
url = "https://picsum.photos/id/101/200/300"
text = "what is this dog name?"
await multi_modal_agent(url,text)

I'm sorry, but there is no dog in the image. The image shows a concrete building with a clear sky in the background.



In [19]:
url = "https://picsum.photos/id/102/200/300"
text = "what is this image about?"
await multi_modal_agent(url,text)

The image shows a few raspberries sitting on a wooden surface, with a blurred, bright background. It appears to be a close-up shot emphasizing the berries.



In [20]:
url = "https://picsum.photos/id/103/200/300"
text = "what is this image about?"
await multi_modal_agent(url,text)

The image shows a person's perspective lying in a field with blue sneakers and jeans, looking out at a grassy field and a blue sky with clouds. It's a relaxing, personal point-of-view shot.



Running and Observing Agents 

In [39]:
from autogen_core import CancellationToken
async def assistant_run()->str:
    response = await assistant.on_messages(
        messages = [TextMessage(content = "Find the information about labrador",source = "user")],
        cancellation_token = CancellationToken(),
    )
    # print(response)
    return response
    


In [40]:
response = await assistant_run()
print("--------------------------inner message ---------------------")

print(response.inner_messages)
print("\n\n\n")
print("--------------------------chat message ---------------------")
print(response.chat_message)

--------------------------inner message ---------------------
[]




--------------------------chat message ---------------------
source='text_agent' models_usage=RequestUsage(prompt_tokens=1719, completion_tokens=38) metadata={} created_at=datetime.datetime(2025, 6, 26, 5, 6, 5, 557268, tzinfo=datetime.timezone.utc) content='Labradors are a popular breed of dog known for their friendly, energetic, and outgoing temperaments. They are intelligent and trainable, making them great family pets and working dogs.\n' type='TextMessage'


In [42]:
async def web_search_agent():
    """search information about the web"""
    return "LaLabradors are a popular breed of dog known for their friendly, energetic, and outgoing temperaments. They are intelligent and trainable, making them great family pets and working dogs."


In [43]:
web_agent = AssistantAgent(
    name = "web_agent",
    model_client = model_client,
    description = "you are a web search agent that can search the web for information",
    system_message = "you are a web search agent that can search the web for information",
    tools = [web_search_agent]
)

In [44]:
from autogen_core import CancellationToken
async def assistant_run()->str:
    response = await web_agent.on_messages(
        messages = [TextMessage(content = "Find the information about labrador",source = "user")],
        cancellation_token = CancellationToken(),
    )
    # print(response)
    return response

In [45]:
response = await assistant_run()
print("--------------------------inner message ---------------------")

print(response.inner_messages)
print("\n\n\n")
print("--------------------------chat message ---------------------")
print(response.chat_message)

--------------------------inner message ---------------------
[ToolCallRequestEvent(source='web_agent', models_usage=RequestUsage(prompt_tokens=28, completion_tokens=5), metadata={}, created_at=datetime.datetime(2025, 6, 26, 5, 9, 31, 224530, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='', arguments='{}', name='web_search_agent')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='web_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 26, 5, 9, 31, 235617, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='LaLabradors are a popular breed of dog known for their friendly, energetic, and outgoing temperaments. They are intelligent and trainable, making them great family pets and working dogs.', name='web_search_agent', call_id='', is_error=False)], type='ToolCallExecutionEvent')]




--------------------------chat message ---------------------
source='web_agent' models_usage=None metadata={} created_at=datetime.dateti

# Streaming Messages

In [48]:
from autogen_agentchat.ui import Console

async def assistant_stream_run()-> None:
    await Console(
        web_agent.on_messages_stream(
        messages = [TextMessage(content = "Find the information about labrador",source = "user")],
        cancellation_token = CancellationToken(),
    ),
    output_stats = True,
    )
await assistant_stream_run()

---------- ToolCallRequestEvent (web_agent) ----------
[FunctionCall(id='', arguments='{}', name='web_search_agent')]
[Prompt tokens: 82, Completion tokens: 5]
---------- ToolCallExecutionEvent (web_agent) ----------
[FunctionExecutionResult(content='LaLabradors are a popular breed of dog known for their friendly, energetic, and outgoing temperaments. They are intelligent and trainable, making them great family pets and working dogs.', name='web_search_agent', call_id='', is_error=False)]
---------- web_agent ----------
LaLabradors are a popular breed of dog known for their friendly, energetic, and outgoing temperaments. They are intelligent and trainable, making them great family pets and working dogs.
---------- Summary ----------
Number of inner messages: 2
Total prompt tokens: 82
Total completion tokens: 5
Duration: 0.85 seconds
